# TP Lecture 3 : Message passing, GCN et GAT

Module 3. Du passage de messages « à la main » à l'entraînement de GNN sur un vrai jeu de données (Cora),
puis deux limites théoriques vues en cours : le **sur-lissage** et la borne d'expressivité de **Weisfeiler--Leman**.

Bibliothèques : `torch`, `torch_geometric`, `networkx`, `numpy`, `matplotlib`.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import networkx as nx
import matplotlib.pyplot as plt
from torch_geometric.nn import GCNConv, GATConv, GINConv, global_add_pool
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import to_undirected
torch.manual_seed(0); np.random.seed(0)

## 1. Message passing à la main
Graphe social à 4 nœuds (attribut = âge). Mise à jour : $h_i\leftarrow\frac12h_i+\frac12\,\mathrm{moyenne}_{j\in\mathcal N(i)}h_j$.
1. Implémentez une étape, puis itérez 30 fois. Que se passe-t-il ?
2. Écrivez la mise à jour sous forme matricielle $h\leftarrow Mh$. Quelle est la limite de $M^kh$ ? Vérifiez-la numériquement.

In [ ]:
G = nx.Graph()
G.add_nodes_from([(0, {"age": 25}), (1, {"age": 30}), (2, {"age": 22}), (3, {"age": 28})])
G.add_edges_from([(0, 1), (1, 2), (2, 3), (3, 0), (0, 2)])
nx.draw(G, nx.spring_layout(G, seed=0), with_labels=True, node_color="lightblue"); plt.show()
# (À compléter)

## 2. Équivariance par permutation
Vérifiez sur une couche `GCNConv` que $f(PAP^\top,PX)=P\,f(A,X)$ pour une permutation $P$ des nœuds.
Indice : si le nouveau nœud $k$ est l'ancien nœud `perm[k]`, les arêtes se renumérotent avec `inv = torch.argsort(perm)`.

In [ ]:
x = torch.tensor([[25.], [30.], [22.], [28.]])
edge_index = to_undirected(torch.tensor([[0, 1, 2, 3, 0], [1, 2, 3, 0, 2]]))
# (À compléter)

## 3. Une couche GCN à la main
Calculez $\tilde D^{-1/2}\tilde A\tilde D^{-1/2}XW^\top+b$ avec les poids d'une `GCNConv` (`conv.lin.weight`, `conv.bias`)
et vérifiez que vous retrouvez sa sortie. Calculez le spectre de $\tilde D^{-1/2}\tilde A\tilde D^{-1/2}$.

In [ ]:
# (À compléter)

## 4. Entraînement sur Cora
`data = Planetoid(root="data", name="Cora")[0]` : 2708 articles, 1433 attributs, 7 classes, 140 nœuds d'entraînement.
1. Entraînez un **MLP** (qui ignore le graphe), un **GCN** à 2 couches et un **GAT** (8 têtes) ; sélectionnez l'époque
   sur l'ensemble de validation et donnez la précision test (moyenne sur 3 graines).
2. Que conclure de la comparaison avec le MLP ?

Remarque : comparer les sorties de deux réseaux **non entraînés** n'aurait aucun sens.

In [ ]:
# (À compléter)

## 5. Sur-lissage
1. Calculez $\hat A^kX$ pour $k$ jusqu'à 1024 et mesurez la dispersion des directions des lignes (lignes normalisées)
   dans la plus grande composante connexe de Cora. Pourquoi se limiter à une composante connexe ?
2. Entraînez des GCN de 2, 4, 8, 16 et 32 couches. Tracez la précision test en fonction de la profondeur.

In [ ]:
# (À compléter)

## 6. Limite d'expressivité
Construisez $C_3\cup C_3$ (deux triangles) et $C_6$ (hexagone), avec le même attribut 1 sur tous les nœuds.
Calculez la représentation de graphe (somme des nœuds) produite par 3 couches `GINConv` : les deux graphes sont-ils
distingués ? Recommencez en donnant comme attribut le nombre de triangles contenant chaque nœud.

In [ ]:
# (À compléter)

## 7. Questions de réflexion
1. Quelle différence observez-vous entre GCN et GAT sur Cora ? Pourquoi ?
2. Dans quels cas l'attention apporte-t-elle un avantage ? Quelles sont ses limites ?
3. Comment adapter ces modèles à de grands graphes, ou à des graphes hétérogènes ?
4. Pourquoi les GNN profonds échouent-ils (section 5) ? Quels remèdes ?
5. Que montre l'expérience de la section 6 ? Est-ce un problème d'entraînement ?